# Preprocesamiento de Datos - Telco Customer Churn

Este notebook carga el dataset Telco Customer Churn, realiza limpieza, codificación de variables categóricas, estandarización de numéricas, y divide en train/test.

**Dataset:** Telco Customer Churn (Kaggle)
**Problema:** Clasificación binaria - predecir si un cliente abandonará la compañía (Churn)

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import joblib
import os

## 1. Carga de datos

In [3]:
df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
print(f'Dimensiones: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')
df.head(3)

Dimensiones: (7043, 21)
Columnas: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 2. Limpieza inicial

In [5]:
# Eliminar columna customerID (no aporta valor predictivo)
df.drop(columns=['customerID'], inplace=True)
print(f'Columnas después de eliminar customerID: {df.shape[1]}')

Columnas después de eliminar customerID: 20


In [6]:
# Convertir TotalCharges a numérico (tiene espacios vacíos)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
nulos = df['TotalCharges'].isna().sum()
print(f'Valores nulos en TotalCharges: {nulos}')

# Imputar con la mediana (es mas robusta que la media para distribuciones con outliers)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print(f'Nulos después de imputación: {df["TotalCharges"].isna().sum()}')

Valores nulos en TotalCharges: 11
Nulos después de imputación: 0


In [7]:
# Verificar nulos en todo el dataset
print('Valores nulos por columna:')
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().any() else 'Sin valores nulos')

Valores nulos por columna:
Sin valores nulos


## 3. Separar features (X) y target (y)

In [8]:
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Distribución de Churn:\n{y.value_counts()}\n')
print(f'Proporción:\n{y.value_counts(normalize=True)}')

X shape: (7043, 19)
y shape: (7043,)
Distribución de Churn:
Churn
0    5174
1    1869
Name: count, dtype: int64

Proporción:
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


## 4. Identificar tipos de columnas

In [9]:
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['str']).columns.tolist()

print(f'Columnas numéricas ({len(numerical_cols)}): {numerical_cols}')
print(f'Columnas categóricas ({len(categorical_cols)}): {categorical_cols}')

Columnas numéricas (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Columnas categóricas (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [10]:
# Ver valores únicos de cada categórica
for col in categorical_cols:
    print(f'{col}: {X[col].nunique()} valores únicos -> {X[col].unique()}')

gender: 2 valores únicos -> <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Partner: 2 valores únicos -> <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Dependents: 2 valores únicos -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
PhoneService: 2 valores únicos -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines: 3 valores únicos -> <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
InternetService: 3 valores únicos -> <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
OnlineSecurity: 3 valores únicos -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
OnlineBackup: 3 valores únicos -> <StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
DeviceProtection: 3 valores únicos -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
TechSupport: 3 valores únicos -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingTV: 3 valores únicos ->

## 5. Construir preprocesador (ColumnTransformer)

In [11]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

## 6. Dividir en train/test (80/20 con estratificación)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')
print(f'\nDistribución train:\n{y_train.value_counts(normalize=True)}')
print(f'\nDistribución test:\n{y_test.value_counts(normalize=True)}')

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)

Distribución train:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Distribución test:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [13]:
# Ajustar preprocesador solo con train
preprocessor.fit(X_train)

# Transformar conjuntos
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f'X_train transformado: {X_train_transformed.shape}')
print(f'X_test transformado: {X_test_transformed.shape}')

X_train transformado: (5634, 30)
X_test transformado: (1409, 30)


In [14]:
# Obtener nombres de las features después del OneHot
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(cat_feature_names)
print(f'Total features después de transformación: {len(all_feature_names)}')
print(f'Primeras 10 features: {all_feature_names[:10]}')

Total features después de transformación: 30
Primeras 10 features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes']


## 7. Guardar artefactos

In [15]:
os.makedirs('../artifacts', exist_ok=True)

joblib.dump(preprocessor, '../artifacts/preprocessor.pkl')
joblib.dump(X_train, '../artifacts/X_train.pkl')
joblib.dump(X_test, '../artifacts/X_test.pkl')
joblib.dump(y_train, '../artifacts/y_train.pkl')
joblib.dump(y_test, '../artifacts/y_test.pkl')

print('Artefactos guardados en artifacts/:')
for f in ['preprocessor.pkl', 'X_train.pkl', 'X_test.pkl', 'y_train.pkl', 'y_test.pkl']:
    path = f'../artifacts/{f}'
    size = os.path.getsize(path) / 1024
    print(f'  {f} - {size:.1f} KB')

Artefactos guardados en artifacts/:
  preprocessor.pkl - 6.7 KB
  X_train.pkl - 390.8 KB
  X_test.pkl - 101.9 KB
  y_train.pkl - 176.8 KB
  y_test.pkl - 44.8 KB


## 8. Resumen

In [16]:
print('=== RESUMEN ===')
print(f'Dataset original: {df.shape[0]} registros, {df.shape[1]+1} columnas (con customerID)')
print(f'Features numéricas: {numerical_cols}')
print(f'Features categóricas: {len(categorical_cols)} columnas')
print(f'Features después de OneHot + Scale: {X_train_transformed.shape[1]}')
print(f'Train: {X_train.shape[0]} muestras ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'Test: {X_test.shape[0]} muestras ({X_test.shape[0]/len(df)*100:.1f}%)')
print(f'Churn rate (train): {y_train.mean():.3f}')
print(f'Churn rate (test): {y_test.mean():.3f}')

=== RESUMEN ===
Dataset original: 7043 registros, 21 columnas (con customerID)
Features numéricas: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Features categóricas: 15 columnas
Features después de OneHot + Scale: 30
Train: 5634 muestras (80.0%)
Test: 1409 muestras (20.0%)
Churn rate (train): 0.265
Churn rate (test): 0.265
